# Signature Pattern Discovery (Data-Driven)

Discover anomaly patterns **directly from training data** — no hand-crafted guesses.

**Approach**:
1. Extract operation fingerprints from every anomaly session
2. Cluster sessions by fingerprint similarity
3. Name each cluster by its dominant operations
4. Generate `HDFS_ERROR_PATTERNS` and `BGL_ERROR_PATTERNS` from real data

Start with **HDFS**, then **BGL**.

## 1. Setup

In [1]:
import sys
import re
from pathlib import Path
from collections import Counter, defaultdict

sys.path.insert(0, str(Path.cwd().parent))

from src.data_loader import HDFSDataLoader, BGLDataLoader, Session

print('Imports OK')

Imports OK


## 2. Load HDFS Training Data

In [2]:
loader = HDFSDataLoader(
    log_file='../logs/HDFS.log',
    label_file='../logs/anomaly_label_HDFS.csv'
)
loader.load()

train_sessions = loader.get_train()
hdfs_anomalies = [s for s in train_sessions if s.label == 1]
hdfs_normals   = [s for s in train_sessions if s.label == 0]

print(f'Training sessions: {len(train_sessions):,}')
print(f'  Anomalies: {len(hdfs_anomalies):,}')
print(f'  Normals:   {len(hdfs_normals):,}')
lengths = [len(s.lines) for s in hdfs_anomalies]
print(f'\nAnomaly line-length: min={min(lengths)}, max={max(lengths)}, median={sorted(lengths)[len(lengths)//2]}')

Loading HDFS logs from: ../logs/HDFS.log
Loading labels from: ../logs/anomaly_label_HDFS.csv


Reading HDFS logs: 11175629it [00:10, 1064323.46it/s]


Found 575061 unique blocks
Training sessions: 402,542
  Anomalies: 11,786
  Normals:   390,756

Anomaly line-length: min=2, max=284, median=20


## 3. Automatic Token Discovery (HDFS)

Instead of hand-picking operation keywords, **discover them from the data**:
1. Extract all words from every session (anomaly + normal sample)
2. Compute per-session presence rate for anomaly vs normal
3. Keep only tokens where the rates differ significantly → **discriminative tokens**
4. These become the fingerprint features

In [ ]:
import random
random.seed(42)
norm_sample = random.sample(hdfs_normals, min(len(hdfs_anomalies), len(hdfs_normals)))

# Step 1: Extract all words from anomaly and normal sessions
anom_word_presence = Counter()   # how many anomaly sessions contain this word
norm_word_presence = Counter()   # how many normal sessions contain this word

for s in hdfs_anomalies:
    text = ' '.join(s.lines).lower()
    words = set(re.findall(r'[a-z]{3,}', text))
    for w in words:
        anom_word_presence[w] += 1

for s in norm_sample:
    text = ' '.join(s.lines).lower()
    words = set(re.findall(r'[a-z]{3,}', text))
    for w in words:
        norm_word_presence[w] += 1

# Step 2: Compute discriminative score for each word
all_words = set(anom_word_presence.keys()) | set(norm_word_presence.keys())
disc_hdfs = []
for w in all_words:
    a_count = anom_word_presence.get(w, 0)
    n_count = norm_word_presence.get(w, 0)
    if a_count < 5 and n_count < 5:
        continue  # skip rare words
    a_rate = a_count / len(hdfs_anomalies)
    n_rate = n_count / len(norm_sample)
    # Discriminative = large absolute difference in rates
    disc = abs(a_rate - n_rate)
    direction = 'ANOM' if a_rate > n_rate else 'NORM'
    disc_hdfs.append((w, a_count, n_count, a_rate, n_rate, disc, direction))

disc_hdfs.sort(key=lambda x: -x[5])

# Step 3: Show top discriminative tokens
print(f'Top discriminative tokens (anomaly vs normal):')
print(f'{"Token":25s} {"Anom sess":>10s} {"Norm sess":>10s} {"Anom%":>7s} {"Norm%":>7s} {"Diff":>7s} {"Dir":>5s}')
print('-' * 75)
for w, ac, nc, ar, nr, disc, d in disc_hdfs[:50]:
    print(f'{w:25s} {ac:10d} {nc:10d} {ar:7.1%} {nr:7.1%} {disc:7.1%} {d:>5s}')

# Step 4: Auto-select discriminative features
# Keep tokens that are significantly more present in anomalies OR normals
# Threshold: |anom_rate - norm_rate| > 0.05 (5% difference)
DISC_THRESHOLD = 0.05
hdfs_features = [(w, ar, nr, d) for w, ac, nc, ar, nr, disc, d in disc_hdfs if disc > DISC_THRESHOLD]

print(f'\n=== Selected {len(hdfs_features)} discriminative features (threshold={DISC_THRESHOLD}) ===')
for w, ar, nr, d in hdfs_features:
    print(f'  {w:25s}  anom={ar:.1%}  norm={nr:.1%}  [{d}]')

Operation             Anom mean  Norm mean  Anom>0%  Norm>0%
----------------------------------------------------------
receiving                  2.72       3.00   100.0%   100.0%
received                   2.14       3.00    62.9%   100.0%
allocate                   1.00       1.00   100.0%   100.0%
addstoredblock             2.64       3.01    62.9%   100.0%
packetresponder            5.69       9.00    63.0%   100.0%
writeblock                 0.20       0.00    19.7%     0.0%
serving                    0.48       0.61    18.8%    22.4%
delete                     2.20       2.44    61.1%    81.4%
replicate                  0.26       0.00    20.5%     0.3%
exception                  0.93       0.61    38.4%    22.4%
terminating                1.89       3.00    63.0%   100.0%
blockrecovery              0.00       0.00     0.0%     0.0%
invalidate                 0.00       0.00     0.0%     0.0%
already                    0.00       0.00     0.0%     0.0%


## 4. Fingerprint with Discovered Tokens

Use the auto-discovered discriminative tokens (not hand-picked) to fingerprint each session.
Binary fingerprint: which discriminative tokens are present?

In [ ]:
# Only use tokens that are more common in anomalies (direction=ANOM)
# Tokens more common in normals indicate "missing normal operations", not error indicators
anom_discriminative = [w for w, ar, nr, d in hdfs_features if d == 'ANOM']
norm_discriminative = [w for w, ar, nr, d in hdfs_features if d == 'NORM']

print(f'Anomaly-enriched tokens ({len(anom_discriminative)}): {anom_discriminative}')
print(f'Normal-enriched tokens  ({len(norm_discriminative)}): {norm_discriminative}')
print(f'(Normal-enriched = operations MISSING in anomalies → structural absence signal)')

# Fingerprint function using discovered tokens
def hdfs_fingerprint_auto(session, features):
    """Binary fingerprint: which discovered tokens are present?"""
    text = ' '.join(session.lines).lower()
    return {f: (1 if f in text else 0) for f in features}

# Use ALL discriminative tokens (both anomaly-enriched and normal-enriched)
# because "missing a normal op" is itself a signal
all_disc_tokens = [w for w, ar, nr, d in hdfs_features]

# Fingerprint all sessions
anom_fps = [(s, hdfs_fingerprint_auto(s, all_disc_tokens)) for s in hdfs_anomalies]
norm_fps = [(s, hdfs_fingerprint_auto(s, all_disc_tokens)) for s in norm_sample]

# Show average fingerprint
print(f'\n{"Token":25s} {"Anom%":>7s} {"Norm%":>7s} {"Signal":>8s}')
print('-' * 50)
for tok in all_disc_tokens:
    a_pct = sum(fp[tok] for _, fp in anom_fps) / len(anom_fps)
    n_pct = sum(fp[tok] for _, fp in norm_fps) / len(norm_fps)
    sig = 'ANOM' if tok in anom_discriminative else 'MISSING'
    print(f'{tok:25s} {a_pct:7.1%} {n_pct:7.1%} {sig:>8s}')

Discovered 13 unique anomaly fingerprints
  # Fingerprint                                                    Count  % Total
--------------------------------------------------------------------------------
  1 addstoredblock+allocate+delete+packetresponder+received+receiving+terminating    3440    29.2%
  2 allocate+exception+receiving+writeblock                         2265    19.2%
  3 allocate+receiving                                              2091    17.7%
  4 addstoredblock+allocate+delete+packetresponder+received+receiving+replicate+terminating    1520    12.9%
  5 addstoredblock+allocate+delete+exception+packetresponder+received+receiving+serving+terminating    1372    11.6%
  6 addstoredblock+allocate+delete+exception+packetresponder+received+receiving+replicate+serving+terminating     832     7.1%
  7 addstoredblock+allocate+packetresponder+received+receiving+terminating     175     1.5%
  8 addstoredblock+allocate+packetresponder+received+receiving+replicate+terminating   

## 5. Cluster by Discovered Fingerprints

Group anomalies by their binary fingerprint over discovered tokens.

In [ ]:
def fingerprint_label(fp):
    """Human-readable label: list tokens that are present."""
    present = sorted(k for k, v in fp.items() if v == 1)
    return '+'.join(present) if present else 'EMPTY'

# Group anomalies by their binary fingerprint
clusters = defaultdict(list)
for s, fp in anom_fps:
    key = fingerprint_label(fp)
    clusters[key].append((s, fp))

sorted_clusters = sorted(clusters.items(), key=lambda x: -len(x[1]))

print(f'Discovered {len(sorted_clusters)} unique anomaly fingerprints (raw)')
print(f'{"#":>3s} {"Fingerprint":70s} {"Count":>7s} {"% Total":>8s}')
print('-' * 90)
for i, (label, members) in enumerate(sorted_clusters, 1):
    pct = len(members) / len(hdfs_anomalies)
    display = label[:68] + '..' if len(label) > 70 else label
    print(f'{i:3d} {display:70s} {len(members):7d} {pct:8.1%}')

covered = sum(len(m) for _, m in sorted_clusters)
print(f'\nTotal: {covered} / {len(hdfs_anomalies)} (100%)')

=== Cluster #1: addstoredblock+allocate+delete+packetresponder+received+receiving+terminating (3440 sessions) ===
  Op counts: {'receiving': 3, 'received': 3, 'allocate': 1, 'addstoredblock': 3, 'packetresponder': 9, 'delete': 4, 'terminating': 3}
  Session: HDFS_blk_6161225427189509238 (20 lines)
    081111 065922 22551 INFO dfs.DataNode$DataXceiver: Receiving block blk_6161225427189509238 src: /10.251.111.80:52524 des
    081111 065922 22649 INFO dfs.DataNode$DataXceiver: Receiving block blk_6161225427189509238 src: /10.251.111.80:55856 des
    081111 065922 34 INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: /user/root/rand6/_temporary/_task_200811101024_
    081111 065923 22695 INFO dfs.DataNode$DataXceiver: Receiving block blk_6161225427189509238 src: /10.251.38.214:34981 des
    081111 065954 22552 INFO dfs.DataNode$PacketResponder: PacketResponder 2 for block blk_6161225427189509238 terminating
    ... (15 more lines)

=== Cluster #2: allocate+exception+receiving+writeblo

## 6. Merge Overlapping Clusters

Raw fingerprints produce too many fine-grained clusters that differ by only ±1 token.
Merge clusters that share the same **core anomaly-enriched tokens** (the operations that make them anomalous).

Strategy: Two fingerprints merge if their anomaly-enriched tokens are identical
(they only differ in normal-enriched tokens, which indicate "how much of the normal pipeline completed").

In [ ]:
# Merge key: only the anomaly-enriched tokens that are present
def merge_key(fp):
    """Merge key = only the anomaly-enriched tokens present in this session."""
    return tuple(sorted(k for k, v in fp.items() if v == 1 and k in anom_discriminative))

def merge_label(mk):
    return '+'.join(mk) if mk else 'NO_ANOM_TOKENS'

# Merge clusters
merged = defaultdict(list)
for s, fp in anom_fps:
    mk = merge_key(fp)
    merged[merge_label(mk)].append((s, fp))

sorted_merged = sorted(merged.items(), key=lambda x: -len(x[1]))

print(f'After merging: {len(sorted_merged)} clusters (was {len(sorted_clusters)} raw)')
print(f'{"#":>3s} {"Core anomaly tokens":55s} {"Count":>7s} {"% Total":>8s} {"Cum%":>7s}')
print('-' * 82)
cum = 0
for i, (label, members) in enumerate(sorted_merged, 1):
    n = len(members)
    pct = n / len(hdfs_anomalies)
    cum += pct
    display = label[:53] + '..' if len(label) > 55 else label
    print(f'{i:3d} {display:55s} {n:7d} {pct:8.1%} {cum:7.1%}')

total_merged = sum(len(m) for _, m in sorted_merged)
print(f'\nTotal: {total_merged} / {len(hdfs_anomalies)} (100%)')

# Also show what normal-enriched tokens are typically missing per cluster
print(f'\n=== What normal ops are missing in each cluster? ===')
for label, members in sorted_merged[:10]:
    # For each cluster, count how often each normal-enriched token is ABSENT
    absent_rates = {}
    for tok in norm_discriminative:
        absent = sum(1 for _, fp in members if fp.get(tok, 0) == 0)
        absent_rates[tok] = absent / len(members)
    missing = [f'{t}({r:.0%})' for t, r in absent_rates.items() if r > 0.3]
    print(f'  {label[:50]:50s}  missing: {", ".join(missing) if missing else "none"}')

Receiving - Received gap distribution:

Anomalies (n=11786):
  gap=  0:   7370 ( 62.5%)
  gap=  1:   2105 ( 17.9%)
  gap=  2:   2271 ( 19.3%)
  gap=  3:     32 (  0.3%)
  gap=  4:      4 (  0.0%)
  gap=  5:      4 (  0.0%)

Normals (sample n=11786):
  gap=  0:  11786 (100.0%)


## 7. Inspect Merged Clusters

Show example sessions from each merged cluster.

In [ ]:
# Show 1 example from each merged cluster
for i, (label, members) in enumerate(sorted_merged[:10], 1):
    s, fp = members[0]
    present_anom = [k for k in anom_discriminative if fp.get(k, 0) == 1]
    missing_norm = [k for k in norm_discriminative if fp.get(k, 0) == 0]
    
    print(f'=== Merged Cluster #{i}: {label} ({len(members)} sessions) ===')
    print(f'  Anomaly tokens present:  {present_anom}')
    print(f'  Normal tokens missing:   {missing_norm}')
    print(f'  Session: {s.session_id} ({len(s.lines)} lines)')
    for line in s.lines[:4]:
        print(f'    {line[:120]}')
    if len(s.lines) > 4:
        print(f'    ... ({len(s.lines) - 4} more lines)')
    print()

Normal session fingerprints: 5 unique
  # Fingerprint                                                    Count  % Total
--------------------------------------------------------------------------------
  1 addstoredblock+allocate+delete+packetresponder+received+receiving+terminating    6951    59.0%
  2 addstoredblock+allocate+delete+exception+packetresponder+received+receiving+serving+terminating    2641    22.4%
  3 addstoredblock+allocate+packetresponder+received+receiving+terminating    2161    18.3%
  4 addstoredblock+allocate+packetresponder+received+receiving+replicate+terminating      31     0.3%
  5 addstoredblock+allocate+delete+exception+packetresponder+received+receiving+replicate+serving+terminating       2     0.0%

Fingerprints unique to anomalies: 8
  allocate+exception+receiving+writeblock                      (2265 sessions)
  allocate+receiving                                           (2091 sessions)
  addstoredblock+allocate+delete+packetresponder+received+receiving

## 8. Generate HDFS_ERROR_PATTERNS from Merged Clusters

Each merged cluster with ≥ 3 sessions becomes a pattern.
Names are derived from the anomaly-enriched tokens in the cluster.

In [ ]:
def auto_name_hdfs(label, members):
    """Name a cluster from its anomaly-enriched tokens. Purely data-driven."""
    tokens = set(label.split('+')) if label != 'NO_ANOM_TOKENS' else set()
    
    # Also compute structural stats from the raw log operations
    # (use simple counts to add context to the name)
    rcv_gaps = []
    for s, fp in members[:100]:
        text = ' '.join(s.lines).lower()
        rcv = text.count('receiving block')
        rcd = text.count('received block')
        rcv_gaps.append(rcv - rcd)
    avg_gap = sum(rcv_gaps) / len(rcv_gaps) if rcv_gaps else 0
    
    # Generate name from token combination
    if not tokens:
        if avg_gap > 0.5:
            return "Incomplete Block Pipeline", f"No anomaly-specific tokens but receiving-received gap={avg_gap:.1f}"
        return "Structural Anomaly (No Unique Tokens)", "Anomaly detected by structural absence of normal operations"
    
    # Build name from the tokens present
    parts = []
    if 'writeblock' in tokens:
        parts.append('Write Pipeline')
    if 'exception' in tokens:
        parts.append('Exception')
    if 'replicate' in tokens:
        parts.append('Replication')
    if 'serving' in tokens:
        parts.append('Block Serving')
    
    # Any remaining tokens not covered
    covered = {'writeblock', 'exception', 'replicate', 'serving'}
    remaining = tokens - covered
    for t in sorted(remaining):
        parts.append(t.capitalize())
    
    name = ' + '.join(parts) if parts else f'Pattern: {label}'
    
    # Build description from what we know
    desc_parts = []
    if 'exception' in tokens:
        desc_parts.append('exception detected')
    if 'writeblock' in tokens:
        desc_parts.append('writeBlock operation present')
    if 'replicate' in tokens:
        desc_parts.append('replication activity')
    if 'serving' in tokens:
        desc_parts.append('block serving activity')
    if avg_gap > 0.5:
        desc_parts.append(f'receiving-received gap={avg_gap:.1f}')
    
    desc = '; '.join(desc_parts) if desc_parts else f'Anomaly with tokens: {label}'
    
    return name, desc

# Generate patterns from merged clusters
print('=== Auto-Generated HDFS Patterns (from merged clusters) ===\n')
hdfs_patterns_discovered = {}
pattern_id = 0

for label, members in sorted_merged:
    if len(members) < 3:
        continue
    
    pattern_id += 1
    name, desc = auto_name_hdfs(label, members)
    
    # Keywords = the anomaly-enriched tokens in this cluster
    tokens = [t for t in label.split('+') if t and t != 'NO_ANOM_TOKENS']
    
    # Also include the normal-enriched tokens that are typically MISSING
    # as "absence indicators" in the description
    absent_norms = []
    for tok in norm_discriminative:
        absent = sum(1 for _, fp in members if fp.get(tok, 0) == 0)
        if absent / len(members) > 0.5:
            absent_norms.append(tok)
    
    key = f'hdfs_{pattern_id:02d}'
    hdfs_patterns_discovered[key] = {
        'name': name,
        'description': desc,
        'keywords': tokens,
        'patterns': [re.escape(t) for t in tokens],
        'frequency': len(members),
        'merge_key': label,
        'typically_missing': absent_norms,
    }
    
    print(f'{pattern_id:2d}. {name}')
    print(f'    Core tokens: {tokens}')
    print(f'    Sessions: {len(members)}')
    print(f'    Description: {desc}')
    print(f'    Usually missing: {absent_norms}')
    print()

total_covered = sum(p['frequency'] for p in hdfs_patterns_discovered.values())
print(f'Patterns: {len(hdfs_patterns_discovered)}')
print(f'Coverage: {total_covered}/{len(hdfs_anomalies)} ({total_covered/len(hdfs_anomalies):.1%})')
uncovered = len(hdfs_anomalies) - total_covered
print(f'Uncovered (< 3 sessions): {uncovered} ({uncovered/len(hdfs_anomalies):.1%})')

=== Auto-Generated HDFS Patterns ===

 1. Block Deletion/Invalidation
    Fingerprint: addstoredblock+allocate+delete+packetresponder+received+receiving+terminating
    Sessions: 3440
    Description: Block deleted or invalidated — unexpected block removal
    Keywords: ['addstoredblock', 'allocate', 'delete', 'packetresponder', 'received', 'receiving', 'terminating']

 2. Write Pipeline Exception
    Fingerprint: allocate+exception+receiving+writeblock
    Sessions: 2265
    Description: Exception during writeBlock — pipeline failure between DataNodes
    Keywords: ['allocate', 'exception', 'receiving', 'writeblock']

 3. Incomplete Block Write
    Fingerprint: allocate+receiving
    Sessions: 2091
    Description: Receiving block without matching Received confirmation (avg gap=1.0)
    Keywords: ['allocate', 'receiving']

 4. Block Deletion/Invalidation
    Fingerprint: addstoredblock+allocate+delete+packetresponder+received+receiving+replicate+terminating
    Sessions: 1520
    Desc

## 9. Export: Python dict for `signature_generator.py`

Print the discovered patterns as Python code you can paste directly into the source.

In [ ]:
# Print as Python code for signature_generator.py
print('# Auto-generated from training data — do not hand-edit')
print('# Discovery notebook: notebooks/05_signature_audit.ipynb')
print(f'# Source: {len(hdfs_anomalies)} training anomalies, {len(hdfs_patterns_discovered)} patterns')
print()
print('HDFS_ERROR_PATTERNS = {')
for key, p in hdfs_patterns_discovered.items():
    print(f'    "{key}": {{')
    print(f'        "name": "{p["name"]}",')
    print(f'        "description": "{p["description"]}",')
    print(f'        "keywords": {p["keywords"]},')
    print(f'        "patterns": {p["patterns"]},')
    print(f'        # frequency={p["frequency"]}, merge_key={p["merge_key"]}')
    print(f'        # typically_missing={p["typically_missing"]}')
    print(f'    }},')
print('}')
print(f'\n# {len(hdfs_patterns_discovered)} patterns, covering {total_covered}/{len(hdfs_anomalies)} anomalies ({total_covered/len(hdfs_anomalies):.1%})')

HDFS_ERROR_PATTERNS = {
    "hdfs_auto_01": {
        "name": "Block Deletion/Invalidation",
        "description": "Block deleted or invalidated — unexpected block removal",
        "keywords": ['addstoredblock', 'allocate', 'delete', 'packetresponder', 'received', 'receiving', 'terminating'],
        "patterns": ['addstoredblock', 'delete', 'packetresponder', 'received block', 'receiving block', 'terminating'],
        # frequency=3440, fingerprint=addstoredblock+allocate+delete+packetresponder+received+receiving+terminating
    },
    "hdfs_auto_02": {
        "name": "Write Pipeline Exception",
        "description": "Exception during writeBlock — pipeline failure between DataNodes",
        "keywords": ['allocate', 'exception', 'receiving', 'writeblock'],
        "patterns": ['exception', 'receiving block', 'writeblock'],
        # frequency=2265, fingerprint=allocate+exception+receiving+writeblock
    },
    "hdfs_auto_03": {
        "name": "Incomplete Block Write",
        "des

---

## 10. Load BGL Training Data

In [ ]:
bgl_loader = BGLDataLoader(log_file='../logs/BGL.log')
bgl_loader.load()

bgl_train = bgl_loader.get_train()
bgl_anomalies = [s for s in bgl_train if s.label == 1]
bgl_normals   = [s for s in bgl_train if s.label == 0]

print(f'BGL Training sessions: {len(bgl_train):,}')
print(f'  Anomalies: {len(bgl_anomalies):,}')
print(f'  Normals:   {len(bgl_normals):,}')

## 11. BGL Keyword Discovery

BGL anomalies are keyword-rich (FATAL, machine check, parity error, etc.).
Extract the most discriminative keywords directly from the data.

In [ ]:
# Extract significant tokens from BGL anomaly lines
# BGL lines have rich error keywords unlike HDFS

# Tokenize and count: which words appear much more in anomalies than normals?
anom_word_counts = Counter()
norm_word_counts = Counter()

for s in bgl_anomalies:
    text = ' '.join(s.lines).lower()
    words = set(re.findall(r'\b[a-z]{3,}\b', text))  # unique words per session
    for w in words:
        anom_word_counts[w] += 1

random.seed(42)
bgl_norm_sample = random.sample(bgl_normals, min(len(bgl_anomalies), len(bgl_normals)))
for s in bgl_norm_sample:
    text = ' '.join(s.lines).lower()
    words = set(re.findall(r'\b[a-z]{3,}\b', text))
    for w in words:
        norm_word_counts[w] += 1

# Compute discriminative score: (anom_rate - norm_rate) * anom_count
# Only consider words appearing in at least 5 anomaly sessions
print(f'{"Word":25s} {"Anom sessions":>14s} {"Norm sessions":>14s} {"Anom %":>8s} {"Norm %":>8s} {"Disc.":>8s}')
print('-' * 75)

disc_scores = []
for word, a_count in anom_word_counts.items():
    if a_count < 5:
        continue
    n_count = norm_word_counts.get(word, 0)
    a_rate = a_count / len(bgl_anomalies)
    n_rate = n_count / max(len(bgl_norm_sample), 1)
    disc = (a_rate - n_rate) * a_count  # high when anomaly-specific and frequent
    disc_scores.append((word, a_count, n_count, a_rate, n_rate, disc))

disc_scores.sort(key=lambda x: -x[5])
for word, a_count, n_count, a_rate, n_rate, disc in disc_scores[:40]:
    print(f'{word:25s} {a_count:14d} {n_count:14d} {a_rate:8.1%} {n_rate:8.1%} {disc:8.1f}')

## 12. Discover & Merge BGL Anomaly Clusters

Same approach as HDFS: use the discriminative words from step 11, fingerprint, cluster, and merge.
For BGL, anomaly tokens are the merge keys (since BGL is already keyword-rich).

In [ ]:
# Auto-select discriminative features for BGL
# Keep words that are anomaly-enriched: anom_rate > norm_rate and difference > 5%
bgl_disc = [(w, ac, nc, ar, nr) for w, ac, nc, ar, nr, d in disc_scores 
            if ar - nr > 0.05 and ar > 0.02]
bgl_features = [w for w, ac, nc, ar, nr in bgl_disc]

print(f'Selected {len(bgl_features)} discriminative BGL features:')
for w, ac, nc, ar, nr in bgl_disc[:20]:
    print(f'  {w:25s}  anom={ar:.1%}  norm={nr:.1%}  diff={ar-nr:.1%}')
if len(bgl_disc) > 20:
    print(f'  ... and {len(bgl_disc) - 20} more')

# Fingerprint BGL anomalies
def bgl_fingerprint(session, features):
    text = ' '.join(session.lines).lower()
    return {f: (1 if f in text else 0) for f in features}

bgl_fps = [(s, bgl_fingerprint(s, bgl_features)) for s in bgl_anomalies]

# Cluster by full fingerprint
bgl_clusters = defaultdict(list)
for s, fp in bgl_fps:
    key = '+'.join(sorted(k for k, v in fp.items() if v == 1)) or 'NONE'
    bgl_clusters[key].append((s, fp))

bgl_sorted_raw = sorted(bgl_clusters.items(), key=lambda x: -len(x[1]))
print(f'\nRaw BGL clusters: {len(bgl_sorted_raw)}')

# For BGL, merging is less critical (keywords are already specific)
# But let's still merge by a "primary error type" heuristic:
# Use only the top-3 most discriminative tokens present as the merge key
# (since BGL fingerprints can have many tokens from overlapping error messages)
top_bgl_tokens = set(bgl_features[:15])  # most discriminative

def bgl_merge_key(fp):
    """Merge key = top discriminative tokens present."""
    present = sorted(k for k, v in fp.items() if v == 1 and k in top_bgl_tokens)
    return '+'.join(present) if present else 'OTHER'

bgl_merged = defaultdict(list)
for s, fp in bgl_fps:
    mk = bgl_merge_key(fp)
    bgl_merged[mk].append((s, fp))

bgl_sorted = sorted(bgl_merged.items(), key=lambda x: -len(x[1]))

print(f'Merged BGL clusters: {len(bgl_sorted)} (from {len(bgl_sorted_raw)} raw)')
print(f'\n{"#":>3s} {"Merge key":55s} {"Count":>7s} {"% Total":>8s} {"Cum%":>7s}')
print('-' * 82)
cum = 0
for i, (label, members) in enumerate(bgl_sorted[:20], 1):
    n = len(members)
    pct = n / len(bgl_anomalies)
    cum += pct
    display = label[:53] + '..' if len(label) > 55 else label
    print(f'{i:3d} {display:55s} {n:7d} {pct:8.1%} {cum:7.1%}')

## 13. Inspect Top BGL Clusters

In [ ]:
# Show 1 example from each top BGL merged cluster
for i, (label, members) in enumerate(bgl_sorted[:10], 1):
    s = members[0][0]
    print(f'=== BGL Cluster #{i}: {label} ({len(members)} sessions) ===')
    for line in s.lines[:3]:
        print(f'    {line[:130]}')
    print()

## 14. Export BGL Patterns as Python dict

In [ ]:
# Generate BGL patterns from merged clusters (>= 3 sessions)
# Auto-name from the cluster's top keywords — no hand-crafted names

bgl_patterns_discovered = {}
pid = 0

for label, members in bgl_sorted:
    if len(members) < 3:
        continue
    pid += 1
    
    keywords = [k for k in label.split('+') if k]
    
    # Auto-generate name: capitalize top 3 keywords → "Keyword1 Keyword2 Error"
    top_kw = keywords[:3]
    name = ' '.join(k.capitalize() for k in top_kw) + ' Error'
    desc = f'Anomaly cluster characterised by: {", ".join(keywords)}'
    
    key = f'bgl_auto_{pid:02d}'
    bgl_patterns_discovered[key] = {
        'name': name,
        'description': desc,
        'keywords': keywords,
        'patterns': [r'\b' + re.escape(k) + r'\b' if len(k) > 3 else k for k in keywords[:5]],
        'frequency': len(members),
        'fingerprint': label,
    }

# Print as Python code
print('BGL_ERROR_PATTERNS = {')
for key, p in bgl_patterns_discovered.items():
    print(f'    "{key}": {{')
    print(f'        "name": "{p["name"]}",')
    print(f'        "description": "{p["description"]}",')
    print(f'        "keywords": {p["keywords"]},')
    print(f'        "patterns": {p["patterns"]},')
    print(f'        # frequency={p["frequency"]}, fingerprint={p["fingerprint"]}')
    print(f'    }},')
print('}')

total_bgl = sum(p['frequency'] for p in bgl_patterns_discovered.values())
print(f'\n# {len(bgl_patterns_discovered)} patterns, covering {total_bgl}/{len(bgl_anomalies)} anomalies ({total_bgl/len(bgl_anomalies):.1%})')

## 15. Summary

**Fully data-driven approach** — no hand-crafted patterns:
1. **Discover** discriminative tokens by comparing anomaly vs normal word frequencies
2. **Fingerprint** every anomaly session using only discovered tokens
3. **Cluster** by fingerprint, then **merge** overlapping clusters (anomaly-enriched tokens as merge key)
4. **Auto-name** each cluster from its dominant keywords
5. **Export** as Python dicts ready for `signature_generator.py`

**Next steps** after reviewing the output:
- Copy the generated `HDFS_ERROR_PATTERNS` and `BGL_ERROR_PATTERNS` into `src/signature_generator.py`
- All patterns are 100% grounded in training data — zero guessing